# Notebook 05 — Informe ejecutivo y conclusiones de negocio
**Proyecto:** A/B Testing y pruebas de hipótesis en AWS SageMaker  
**Objetivo:** convertir los análisis estadísticos de los notebooks 01–04 en una decisión de negocio clara, reproducible y documentada.

> Este notebook **no asume que el experimento fue exitoso**. Calcula los resultados sobre el dataset disponible. Distingue significancia estadística de relevancia económica y describe las limitaciones de los datos limpiados.

**Entradas:** `processed/ab_data_clean.csv` (obligatoria); opcionales `results/notebook03/hypothesis_test_results.csv` y `results/notebook04/statistical_summary.json`.  
**Salidas:** `results/notebook05/` en S3 o `outputs/notebook05/` localmente.

## 1. Preparación del entorno
En SageMaker JupyterLab, utiliza kernel Python 3. Si faltan paquetes, ejecuta `%pip install pandas numpy scipy statsmodels matplotlib boto3` en una celda y reinicia el kernel. Completa la configuración de S3. Para una ejecución local, cambia `USE_S3 = False` y coloca el CSV limpio en la ruta indicada.

In [ ]:
from pathlib import Path
from io import BytesIO
from datetime import datetime, timezone
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import binomtest
from statsmodels.stats.proportion import proportions_ztest, confint_proportions_2indep
from IPython.display import display, Markdown

BUCKET = "TU_BUCKET"  # nombre real del bucket
USE_S3 = True
INPUT_KEY = "processed/ab_data_clean.csv"
LOCAL_PATH = Path("data/ab_data_clean.csv")
OUTPUT_DIR = Path("outputs/notebook05")
OUTPUT_PREFIX = "results/notebook05/"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

ALPHA = 0.05
EXPECTED_TREATMENT_SHARE = 0.5  # verificar contra el diseño real
# Supuestos de negocio: EDITAR antes de presentar un informe real
VISITANTES_MENSUALES = 500_000
MARGEN_POR_CONVERSION_MXN = 150.0
COSTO_IMPLEMENTACION_MXN = 100_000.0
COSTO_OPERATIVO_MENSUAL_MXN = 0.0
HORIZONTE_MESES = 3
MDE_ABSOLUTA = 0.01 # mejora mínima relevante = 1 punto porcentual

s3 = None
if USE_S3:
    if BUCKET == 'TU_BUCKET':
        raise ValueError('Configura BUCKET o establece USE_S3=False')
    import boto3
    s3 = boto3.client('s3')

## 2. Cargar y validar el dataset limpio
Las comprobaciones de integridad evitan decisiones basadas en archivos incorrectos. Se requiere una observación por usuario en esta versión educativa.

In [ ]:
if USE_S3:
    obj = s3.get_object(Bucket=BUCKET, Key=INPUT_KEY)
    df = pd.read_csv(BytesIO(obj['Body'].read()))
else:
    df = pd.read_csv(LOCAL_PATH)

required = {'user_id', 'group', 'landing_page', 'converted'}
missing = required - set(df.columns)
if missing:
    raise ValueError(f'Faltan columnas: {missing}')
if df[list(required)].isna().any().any():
    raise ValueError('Hay valores faltantes en variables críticas')
if not df['user_id'].is_unique:
    raise ValueError('Los registros no son únicos por user_id')
if not df['group'].isin(['control', 'treatment']).all():
    raise ValueError('Grupos inválidos')
if not df['converted'].isin([0,1]).all():
    raise ValueError('converted debe ser 0 o 1')
consistent = ((df['group'].eq('control') & df['landing_page'].eq('old_page')) |
              (df['group'].eq('treatment') & df['landing_page'].eq('new_page')))
if not consistent.all():
    raise ValueError('Aún hay asignaciones inconsistentes en el archivo limpio')
if set(df['group']) != {'control', 'treatment'}:
    raise ValueError('Se requieren ambos grupos')
print(f'Registros analizados: {len(df):,}')
display(df.head())

## 3. Resumen ejecutivo de conversiones
Estimamos la tasa de conversión por grupo, la diferencia B−A en puntos porcentuales y el *lift* relativo. El cambio absoluto determina las conversiones incrementales potenciales.

In [ ]:
summary = df.groupby('group')['converted'].agg(users='size', conversions='sum', rate='mean').reindex(['control','treatment'])
n_a, n_b = map(int, summary['users'].to_numpy())
x_a, x_b = map(int, summary['conversions'].to_numpy())
p_a, p_b = map(float, summary['rate'].to_numpy())
diff = p_b - p_a
lift = (p_b / p_a - 1) if p_a > 0 else np.nan
summary['rate_pct'] = summary['rate'] * 100
print(f'Diferencia B - A: {diff*100:+.4f} puntos porcentuales')
print(f'Lift relativo: {lift*100:+.3f}%' if np.isfinite(lift) else 'Lift no definido')
display(summary)

## 4. Prueba Z, intervalo de confianza y SRM
Aplicamos una prueba bilateral de dos proporciones con α = 0.05. El IC Newcombe de 95% cuantifica la diferencia compatible con los datos. Comprobamos *Sample Ratio Mismatch* respecto de una asignación esperada 50/50: **si el diseño original no fue 50/50, cambia ese parámetro**.

La ausencia de SRM **no demuestra** que la aleatorización haya funcionado. Las exclusiones del Notebook 01 pueden introducir sesgo.

In [ ]:
z_stat, p_value = proportions_ztest([x_b,x_a],[n_b,n_a],alternative='two-sided')
ci_lo,ci_hi = confint_proportions_2indep(x_b,n_b,x_a,n_a,compare='diff',method='newcomb',alpha=ALPHA)
srm_p = binomtest(n_b, n_a+n_b, p=EXPECTED_TREATMENT_SHARE).pvalue
stats_result = pd.DataFrame([{
    'n_control':n_a,'n_treatment':n_b,'rate_control':p_a,'rate_treatment':p_b,
    'difference_B_minus_A':diff,'relative_lift':lift,
    'z_statistic':z_stat,'p_value_two_sided':p_value,
    'ci95_newcombe_lower':ci_lo,'ci95_newcombe_upper':ci_hi,'srm_p_value':srm_p,
    'alpha':ALPHA
}])
display(stats_result.T)
print('Rechazar H0:' if p_value < ALPHA else 'No rechazar H0:', p_value < ALPHA)
if srm_p < ALPHA:
    print('ALERTA: posible Sample Ratio Mismatch; investigar antes de decidir.')

## 5. Consistencia con los notebooks anteriores (opcional)
Si existen los resultados de los notebooks 03 y 04, podemos comparar las métricas clave. **No dependen** de que estos artefactos estén presentes: se recalculan arriba desde el dataset limpio.

In [ ]:
def optional_read(key, local_path, reader):
    try:
        if USE_S3:
            content = s3.get_object(Bucket=BUCKET, Key=key)['Body'].read()
            return reader(BytesIO(content))
        p = Path(local_path)
        return reader(p) if p.exists() else None
    except Exception as exc:
        print(f'Archivo opcional no disponible ({key}): {type(exc).__name__}')
        return None

previous03 = optional_read('results/notebook03/hypothesis_test_results.csv',
                           'outputs_notebook_03/hypothesis_test_results.csv',pd.read_csv)
previous04 = optional_read('results/notebook04/statistical_summary.json',
                           'outputs/notebook04/statistical_summary.json',
                           lambda path: json.loads(path.read().decode('utf-8') if hasattr(path,'read') else Path(path).read_text(encoding='utf-8')))
if previous03 is not None and not previous03.empty:
    old_p = float(previous03.iloc[0]['p_value_two_sided'])
    print(f'Notebook03 p={old_p:.8g} | Notebook05 p={p_value:.8g} | Coinciden:',np.isclose(old_p,p_value,rtol=1e-5,atol=1e-10))
if isinstance(previous04,dict):
    print('IC bootstrap Notebook04:', previous04.get('bootstrap_ci_lower'), previous04.get('bootstrap_ci_upper'))

## 6. Visualizaciones para el informe
Una gráfica muestra las tasas A/B; otra, el efecto estimado y su intervalo de confianza. Escalamos la diferencia a puntos porcentuales para evitar confusiones.

In [ ]:
fig, ax = plt.subplots(figsize=(7,4))
ax.bar(['Control (A)','Tratamiento (B)'],[p_a*100,p_b*100],color=['#64748b','#0d9488'])
for i,value in enumerate([p_a*100,p_b*100]):
    ax.text(i,value+0.15,f'{value:.3f}%',ha='center')
ax.set_ylabel('Tasa de conversión (%)')
ax.set_ylim(0,max(p_a,p_b)*100*1.20)
ax.set_title('Conversión por variante')
fig.tight_layout();fig.savefig(OUTPUT_DIR/'conversion_by_group.png',dpi=170);plt.show()

fig,ax=plt.subplots(figsize=(8,2.8))
ax.errorbar(diff*100,0,xerr=[[max(0,(diff-ci_lo)*100)],[max(0,(ci_hi-diff)*100)]],fmt='o',capsize=6,color='#0d9488')
ax.axvline(0,color='#dc2626',linestyle='--',label='Sin efecto')
ax.set_yticks([]);ax.set_xlabel('Diferencia B − A (puntos porcentuales)')
ax.set_title('Efecto observado e IC del 95%');ax.legend()
fig.tight_layout();fig.savefig(OUTPUT_DIR/'effect_and_ci.png',dpi=170);plt.show()

## 7. Impacto económico: escenario ilustrativo, no pronóstico garantizado
Personaliza tráfico, margen y costos. El cálculo extrapola la diferencia observada a **todos** los visitantes mensuales suponiendo que el efecto se conserva al desplegar. Incluimos el intervalo de confianza para expresar la incertidumbre. **No equivalen a ventas observadas ni corrigen problemas de causalidad.**

In [ ]:
def business_case(delta):
    incremental_conversions = VISITANTES_MENSUALES * delta
    monthly_margin = incremental_conversions * MARGEN_POR_CONVERSION_MXN
    net_horizon = HORIZONTE_MESES * (monthly_margin - COSTO_OPERATIVO_MENSUAL_MXN) - COSTO_IMPLEMENTACION_MXN
    return incremental_conversions,monthly_margin,net_horizon

rows=[]
for label,effect in [('IC inferior',ci_lo),('Estimación',diff),('IC superior',ci_hi)]:
    extra,month,net=business_case(effect)
    rows.append({'escenario':label,'delta_pp':effect*100,'conversiones_adicionales_mes':extra,
                 'margen_incremental_mes_MXN':month,'beneficio_neto_horizonte_MXN':net})
impact=pd.DataFrame(rows)
display(impact.style.format({'delta_pp':'{:+.3f}', 'conversiones_adicionales_mes':'{:,.0f}',
                             'margen_incremental_mes_MXN':'${:,.0f}', 'beneficio_neto_horizonte_MXN':'${:,.0f}'}))
print('Supuestos:',{'visitantes_mes':VISITANTES_MENSUALES,'margen_por_conversion_MXN':MARGEN_POR_CONVERSION_MXN,
                  'implementacion_MXN':COSTO_IMPLEMENTACION_MXN,'costos_mes_MXN':COSTO_OPERATIVO_MENSUAL_MXN,'meses':HORIZONTE_MESES})

## 8. Regla de decisión y límites del análisis
La decisión combina estadística, tamaño del efecto, economía y calidad del experimento:

- **Posible despliegue sujeto a validación:** mejora estadísticamente significativa, estimación positiva, supera MDE y el caso económico central es favorable; sin alertas graves de calidad.
- **Mantener control / investigar:** efecto estadísticamente negativo o SRM detectado.
- **Resultado inconcluso:** no se rechaza H₀, o el efecto es demasiado pequeño para ser relevante económicamente. Un resultado no significativo no prueba equivalencia.

La regla es una guía, **no** una autorización automática de despliegue. Falta revisar el diseño original, exclusiones, segmentación predefinida y métricas de seguridad.

In [ ]:
significant=p_value<ALPHA
srm_alert=srm_p<ALPHA
business_positive=business_case(diff)[2]>0
if srm_alert:
    decision='INVESTIGAR: desbalance de asignación (SRM), no desplegar solo por este análisis.'
elif significant and ci_hi<0:
    decision='MANTENER CONTROL: evidencia de caída de conversión.'
elif significant and ci_lo>0 and diff>=MDE_ABSOLUTA and business_positive:
    decision='CANDIDATO A DESPLIEGUE, pendiente de validar supuestos, exclusiones y guardrails.'
elif significant and ci_lo>0:
    decision='MEJORA ESTADÍSTICA, pero no cumple todos los criterios comerciales fijados.'
else:
    decision='INCONCLUSO: no hay evidencia suficiente de diferencia; no afirmar equivalencia.'
print(decision)
print('Limitación crítica: se utiliza muestra limpia/excluida del Notebook 01; investigar sesgo de selección y análisis intention-to-treat.')

## 9. Generar informe ejecutivo automático en Markdown
El archivo contiene los valores reales calculados durante esta ejecución. Los supuestos de negocio quedan etiquetados claramente.

In [ ]:
report=f"""# Informe ejecutivo — Experimento A/B

**Fecha de generación (UTC):** {datetime.now(timezone.utc).strftime('%Y-%m-%d %H:%M')}  
**Fuente:** {'s3://'+BUCKET+'/'+INPUT_KEY if USE_S3 else str(LOCAL_PATH)}  
**Método:** prueba Z bilateral de dos proporciones, IC 95% Newcombe, α={ALPHA:.2f}.

## Resumen ejecutivo
Se analizaron {n_a+n_b:,} usuarios: {n_a:,} en A y {n_b:,} en B. La conversión fue {p_a:.3%} en A y {p_b:.3%} en B, una diferencia B−A de {diff*100:+.3f} puntos porcentuales (lift relativo {lift*100:+.2f}% si definido).

## Evidencia estadística
- Estadístico Z: {z_stat:.4f}
- p-value bilateral: {p_value:.6g}
- IC del 95% de la diferencia: [{ci_lo*100:+.3f}, {ci_hi*100:+.3f}] puntos porcentuales
- SRM: p={srm_p:.6g} (comparación con asignación esperada de {EXPECTED_TREATMENT_SHARE:.0%} a B)
- Se {'rechaza' if significant else 'no se rechaza'} H₀ al 5%. Esto no demuestra por sí solo relevancia comercial.

## Impacto económico (supuestos editables)
- Visitantes únicos mensuales: {VISITANTES_MENSUALES:,}
- Margen por conversión incremental: ${MARGEN_POR_CONVERSION_MXN:,.2f} MXN
- Costo de implementación: ${COSTO_IMPLEMENTACION_MXN:,.2f} MXN
- Costo operativo mensual: ${COSTO_OPERATIVO_MENSUAL_MXN:,.2f} MXN
- Horizonte: {HORIZONTE_MESES} meses
- Conversiones adicionales esperadas por mes (estimación puntual): {business_case(diff)[0]:+,.0f}
- Margen incremental mensual (estimación puntual): ${business_case(diff)[1]:+,.2f} MXN
- Beneficio neto en horizonte (estimación puntual): ${business_case(diff)[2]:+,.2f} MXN
- Rango del beneficio neto basado en IC del efecto: ${business_case(ci_lo)[2]:+,.2f} a ${business_case(ci_hi)[2]:+,.2f} MXN

Estos importes son escenarios condicionales a que el efecto se mantenga y a que los supuestos sean válidos; no son ingresos realizados.

## Recomendación
**{decision}**

## Limitaciones y seguimiento
1. El dataset corresponde a observaciones depuradas; las exclusiones pueden causar sesgo. Contrastar con una estimación por asignación original (intention-to-treat) cuando se disponga de esa información.
2. Verificar aleatorización original, SRM, unidad de análisis y medición de conversiones.
3. Definir criterios de éxito, MDE y métricas de guardrail antes de nuevos experimentos.
4. Evitar interpretar p≥0.05 como prueba de igualdad.
5. Confirmar las hipótesis comerciales antes de estimar retornos.
"""
report_path=OUTPUT_DIR/'executive_report.md'
report_path.write_text(report,encoding='utf-8')
display(Markdown(report))

## 10. Exportar resultados a S3
Guarda las tablas y el informe; si `USE_S3=True` se subirán a `results/notebook05/`. No se sobrescribe el dataset de origen.

In [ ]:
summary.reset_index().to_csv(OUTPUT_DIR/'conversion_summary.csv',index=False)
stats_result.to_csv(OUTPUT_DIR/'statistical_results.csv',index=False)
impact.to_csv(OUTPUT_DIR/'business_impact_scenarios.csv',index=False)
metadata={'alpha':ALPHA,'expected_treatment_share':EXPECTED_TREATMENT_SHARE,
          'mde_absolute':MDE_ABSOLUTA,'source':INPUT_KEY if USE_S3 else str(LOCAL_PATH),
          'decision':decision, 'business_assumptions':{
          'monthly_visitors':VISITANTES_MENSUALES,'margin_per_conversion_mxn':MARGEN_POR_CONVERSION_MXN,
          'implementation_cost_mxn':COSTO_IMPLEMENTACION_MXN,'monthly_operating_cost_mxn':COSTO_OPERATIVO_MENSUAL_MXN,
          'months':HORIZONTE_MESES}}
(OUTPUT_DIR/'metadata.json').write_text(json.dumps(metadata,ensure_ascii=False,indent=2),encoding='utf-8')
if USE_S3:
    for path in sorted(OUTPUT_DIR.iterdir()):
        if path.is_file():
            key=f'{OUTPUT_PREFIX}{path.name}'
            s3.upload_file(str(path),BUCKET,key)
            print(f's3://{BUCKET}/{key}')
else:
    print('Resultados locales:',OUTPUT_DIR.resolve())

## 11. Ejercicios para reforzar tu criterio estadístico
1. Cambia `MDE_ABSOLUTA` de 0.01 a 0.005. ¿La recomendación comercial cambia aunque el p-value sea el mismo?
2. Reduce el margen de contribución por conversión un 50%. ¿El escenario económico sigue siendo favorable?
3. Compara el IC Newcombe con el bootstrap del Notebook 04. ¿Qué diferencias observas?
4. ¿Por qué es incorrecto interpretar p≥0.05 como “las páginas convierten igual”?
5. ¿Qué pasaría si el grupo B muestra SRM? Enumera posibles fallas instrumentales y de asignación.
6. ¿Qué datos adicionales requerirías para hacer un análisis intention-to-treat?

**Entregables de portafolio:** tablas, dos figuras y el informe `executive_report.md`. Documenta supuestos, decisiones y limitaciones en el README de GitHub.